In [0]:
from pyspark.sql.functions import col,to_date, regexp_replace
import pyspark.sql.functions as F

bronze_df = spark.table("workspace.retail.bronze_retail_sales")

silver_df = bronze_df \
    .withColumn("transaction_id", col("transaction_id").cast("int")) \
    .withColumn("customer_id",regexp_replace("customer_id", "^CUST", "")) \
    .withColumn("customer_id", col("customer_id").cast("int")) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
    .withColumn("age", col("age").cast("int")) \
    .withColumn("quantity", col("quantity").cast("int")) \
    .withColumn("price_per_unit", col("price_per_unit").cast("double")) \
    .withColumn("total_amount", col("total_amount").cast("double"))


In [0]:
# Check nulls
silver_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in silver_df.columns
]).show()

silver_df.printSchema()
silver_df.describe()

In [0]:
# Create table
spark.sql("DROP TABLE IF EXISTS workspace.retail.silver_retail_sales")

silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.retail.silver_retail_sales")